# TradingAgents #1300: a live snapshot inside a 2024 analysis

https://github.com/TauricResearch/TradingAgents/issues/1300

`get_fundamentals` stamped its header with `datetime.now()`. For a symbol with no statements the
whole payload was a market snapshot dated 2026-09-04 inside a run dated 2024-05-10.

In [1]:
import fin_pit as pit
from fin_pit.adapters import yfinance as yfa

info = {"marketCap": 1_200_000_000_000}   # synthetic

def get_fundamentals(info):
    return f"# Data retrieved on: 2026-09-04\nmarketCap: {info['marketCap']}"

print(get_fundamentals(info))   # nothing stops it

# Data retrieved on: 2026-09-04
marketCap: 1200000000000


## With fin-pit

In [2]:
guarded = pit.guard(get_fundamentals, scan_text=True)
snapshot = pit.guard(lambda: yfa.snapshot(info, retrieved_at="2026-09-04T14:00Z"))

with pit.asof("2024-05-10 16:00", tz="America/New_York"):
    for fn, args in ((guarded, (info,)), (snapshot, ())):
        try:
            fn(*args)
        except pit.LookaheadError as e:
            print("refused:", e)

refused: 1 date(s) in the output are later than the decision time 2024-05-10T20:00:00+00:00: 2026-09-04
refused: 1 record is not available at the decision time 2024-05-10T20:00:00+00:00 (first available_at: 2026-09-04T14:00:00+00:00)
